#Chile - Universidad Adolfo Ibáñez (UAI)
##Curso NLP
### Text Classification - Document Classification (Inglés)

In [1]:
#Instalar Librerías
!pip install datasets scikit-learn pandas -q
!pip install --upgrade transformers -q

In [2]:
#Importar Librerías
import pandas as pd
from datasets import Dataset
import torch

In [3]:
# Load dataset
df = pd.read_csv("pubmed_multilabel_classification.csv")
df.columns = ["Title", "Psych", "Phenomena", "Tech", "Health"]

In [4]:
#Lista de Labels
df["labels"] = df[["Psych", "Phenomena", "Tech", "Health"]].values.tolist()
df = df[["Title", "labels"]]
df.loc[:, "labels"] = df["labels"].apply(lambda x: [float(i) for i in x])

In [5]:
#Definir Labels & Hugging Face Dataset
label_names = ["Psych", "Phenomena", "Tech", "Health"]
num_labels = len(label_names)
dataset = Dataset.from_pandas(df)

In [6]:
#Importar Librerías
from transformers import AutoTokenizer
from transformers import AutoModelForSequenceClassification

In [7]:
#Crear Tokenizador
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [8]:
#Función de Tokenización
def tokenize(batch):
    return tokenizer(batch["Title"], padding=True, truncation=True)

In [9]:
#Tokenización & Formateo Labels
dataset = dataset.map(tokenize, batched=True)

Map:   0%|          | 0/1500 [00:00<?, ? examples/s]

In [10]:
#Train & Test Split
dataset = dataset.train_test_split(test_size=0.2)

In [11]:
#Creación de Modelo
model = AutoModelForSequenceClassification.from_pretrained(
    "bert-base-uncased",
    num_labels=num_labels,
    problem_type="multi_label_classification"
)

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [12]:
#Importar Librerías
from transformers import TrainingArguments, Trainer
from sklearn.metrics import f1_score, accuracy_score
import numpy as np

In [13]:
#Desahiblitar Weights & Biases
import os
os.environ["WANDB_DISABLED"] = "true"

In [14]:
#Configuración Entrenamiento
args = TrainingArguments(
    output_dir="pubmed-multilabel",
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=3,
    report_to="none"
)

In [15]:
#Función de Evaluación
def compute_metrics(pred):
    logits, labels = pred
    preds = (logits > 0).astype(int)
    f1_weighted = f1_score(labels, preds, average="weighted")
    f1_per_class = f1_score(labels, preds, average=None)
    return {"Weighted F1": f1_weighted, "Per-Class F1": f1_per_class}

In [16]:
#Importar Librerías
from transformers import DataCollatorWithPadding

In [17]:
#Agregar DataCollator
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

In [18]:
#Parametrizar Entrenamiento
trainer = Trainer(
    model=model,
    args=args,
    train_dataset=dataset["train"],
    eval_dataset=dataset["test"],
    compute_metrics=compute_metrics,
    data_collator=data_collator
)

In [19]:
#Entrenar Modelo
trainer.train()

Step,Training Loss


TrainOutput(global_step=450, training_loss=0.36282955593532984, metrics={'train_runtime': 75.8416, 'train_samples_per_second': 47.467, 'train_steps_per_second': 5.933, 'total_flos': 142452527817600.0, 'train_loss': 0.36282955593532984, 'epoch': 3.0})

In [20]:
#Evaluar Modelo
results = trainer.evaluate()

In [21]:
#Revisar Resultados
print("Weighted F1:", results['eval_Weighted F1'])
print("Per-Class F1:", results['eval_Per-Class F1'])

Weighted F1: 0.7312545646947769
Per-Class F1: [0.69902913 0.82539683 0.28070175 0.73825503]


In [22]:
#Utilizar el Modelo en Modo Inferencia
new_article = "Overview of the multiview and 3D extensions of high efficiency video coding"

# Tokenizar el texto
inputs = tokenizer(new_article, return_tensors="pt", truncation=True, padding=True)

In [23]:
#Mover Inputs a GPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
inputs = {key: value.to(device) for key, value in inputs.items()}

In [24]:
# Usar el modelo en modo inferencia
model.eval()
with torch.no_grad():
    outputs = model(**inputs)
    logits = outputs.logits
    probs = logits.sigmoid()
    prediction = (probs > 0.5).int()

In [25]:
#Revisar Resultados
print("Predicción Binaria:", prediction.tolist())
print("Probabilidades (Continuas):", probs.tolist())

Predicción Binaria: [[0, 1, 1, 1]]
Probabilidades (Continuas): [[0.0265823844820261, 0.8277681469917297, 0.5292912125587463, 0.7651835680007935]]
